In [1]:

import jax
import jax.numpy as jnp
import flax.nnx as nnx
import netket as nk
import netket.experimental as nkx
from NES_VMC import NESTotalAnsatz, create_machine,init_sampler_state,\
    generate_random_initial_states,ha,SingleStateAnsatz,create_single_machine,\
        create_machine_matrix,Ham_psi,Ham_Psi,NES_loss_energy,nes_vmc_gradient,hi,\
        NESTotalAnsatz
import optax
from typing import Callable
from functools import partial
from jax.flatten_util import ravel_pytree
K=2
hi_ext = hi**K

# 2. 初始化 NESTotalAnsatz 模型
rngs = nnx.Rngs(42)
total_ansatz = NESTotalAnsatz(4, n_states=K, hidden_dim=8, rngs=rngs)
single_ansatz = SingleStateAnsatz(4, hidden_dim=8, rngs=rngs)
total_machine, total_graphdef, total_params = create_machine(total_ansatz)
total_matrix_machine, total_graphdef, total_params = create_machine_matrix(total_ansatz)

single_machine_list = []
for ansatz in total_ansatz.single_ansatz_list:
    m, g, p = create_single_machine(ansatz)
    single_machine_list.append(m)
#
samples,keys = init_sampler_state(hi=hi_ext,n_chains=20,seed=12)

# 4. 初始化优化器（和原代码一致）
optimizer = optax.sgd(learning_rate=0.01)
opt_state = optimizer.init(total_params)

/opt/miniconda3/envs/Netket/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


∣NK⟩ Tip: To build H|ψ⟩ use nk.vqs.apply_operator(H, vstate_ψ).

H₂ FCI 基准能量
E0 = -1.01546825 Ha  |  激发能：0.0000 eV
E1 = -0.87542794 Ha  |  激发能：3.8107 eV
E2 = -0.42938376 Ha  |  激发能：15.9482 eV
E3 = -0.26922131 Ha  |  激发能：20.3064 eV


In [ ]:
class NESTotalAnsatz(nnx.Module):
    def __init__(self, n_spin_orbitals: int, n_states: int = 2, hidden_dim: int = 8, *, rngs: nnx.Rngs):
        super().__init__()
        self.K = n_states
        self.n_spin = n_spin_orbitals

        self.single_ansatz_list = nnx.List()
        key = rngs.params()
        for _ in range(n_states):
            key, sub_key = jax.random.split(key)
            sub_rngs = nnx.Rngs(params=sub_key)
            
            ansatz = SingleStateAnsatz(
                n_spin_orbitals, 
                hidden_dim, 
                rngs=sub_rngs
            )
            self.single_ansatz_list.append(ansatz)
    def __call__(self, x: jax.Array):
        def _forward_single(x_single):
            # 形状：[K, n_spin]
            #print(f'x_single.shape: {x_single.shape}')
            x_single = x_single.reshape(self.K, self.n_spin)
            L = jnp.zeros((self.K, self.K), dtype=complex)
            for i in range(self.K):
                for j in range(self.K):
                    L = L.at[i, j].set(
                        self.single_ansatz_list[j](x_single[i])
                    )
                    
            sign, log_abs_det = jnp.linalg.slogdet(jnp.exp(L))
            log_Psi = log_abs_det + 1j * jnp.angle(sign)
            return log_Psi, L  
        
        # 安全的批量处理
        if x.ndim == 2 and x.shape[-1] == self.n_spin:
            # 直接处理单个样本
            return _forward_single(x)
        elif x.ndim == 2 and x.shape[-1] == self.n_spin*self.K:
            x = x.reshape(-1, self.K, self.n_spin)
            # 直接处理批量样本
            return jax.vmap(_forward_single)(x)
        
        elif x.ndim == 3:
            x = x.reshape(-1, self.K, self.n_spin)
            return jax.vmap(_forward_single)(x)
        elif x.ndim ==1:
            x = x[None, :]
            x = x.reshape(self.K, self.n_spin)
            return _forward_single(x)
        else:
            raise ValueError(f'不支持的输入形状: {x.shape}')
            
    

In [ ]:
rngs = nnx.Rngs(42)
total_ansatz = NESTotalAnsatz(4, n_states=K, hidden_dim=8, rngs=rngs)
single_ansatz = SingleStateAnsatz(4, hidden_dim=8, rngs=rngs)
total_machine, total_graphdef, total_params = create_machine(total_ansatz)
total_matrix_machine, total_graphdef, total_params = create_machine_matrix(total_ansatz)

single_machine_list = []
for ansatz in total_ansatz.single_ansatz_list:
    m, g, p = create_single_machine(ansatz)
    single_machine_list.append(m)

In [ ]:
total_machine(total_params,hi_ext.all_states()[1])

In [ ]:
log_Psi , log_M = total_ansatz(hi_ext.all_states()[1])
print(log_Psi,log_M)


In [ ]:
jnp.log(jnp.linalg.det(M))

In [ ]:
eig_vals, eig_vecs = jnp.linalg.eigh(M)
eig_vals

In [ ]:
single_samples= samples.reshape(-1,2,4)
Ham_psi(ha, single_machine_list[0], total_params['single_ansatz_list'][1], single_samples[0][0])

In [ ]:
Ham_Psi(ha=ha,
        single_machine_list=single_machine_list,
        total_params=total_params,
        x=single_samples) #single_samples[0].shape = (2,4)

In [ ]:
single_samples[0:2]

In [ ]:
trace_loss, E_L = NES_loss_energy(ha=ha, 
                total_matrix_machine = total_matrix_machine,
                single_machine_list = single_machine_list,
                total_params = total_params,
                x = single_samples[0:2])

In [ ]:
from jax.flatten_util import ravel_pytree
import jax.numpy as jnp
import jax

def compute_nes_qgt(total_machine, params, samples, diag_shift=0.01):
    # 1. 单样本梯度
    def _single_grad(x):
        return jax.grad(lambda p: total_machine(p, x), holomorphic=True)(params)

    # 2. 对每个样本求导
    grads = [_single_grad(x) for x in samples]  # 列表 [B]

    # 3. 展平每个梯度 → list of [P]
    flat_grads = [ravel_pytree(g)[0] for g in grads]

    # 4. 堆叠成 → (B, P)
    grads_flat = jnp.stack(flat_grads)  

    # 5. 计算 QGT
    g = grads_flat
    g_conj = g.conj()

    term1 = jnp.mean(g_conj[:, :, None] * g[:, None, :], axis=0)
    g_mean = jnp.mean(g, axis=0)
    term2 = g_mean.conj()[:, None] * g_mean[None, :]
    S = term1 - term2

    S_reg = S + diag_shift * jnp.eye(S.shape[0], dtype=complex)
    unravel = ravel_pytree(params)[1]

    return S_reg, unravel


S_reg, unravel_fn = compute_nes_qgt(
    total_machine, 
    total_params, 
    single_samples[0:2]
)
print(S_reg.shape)  # ✅ 完美输出 (P, P)

In [ ]:
grad, loss_mean, E_L_mean = nes_vmc_gradient(
    ha=ha,
    total_matrix_machine=total_matrix_machine,
    total_machine=total_machine,
    single_machine_list=single_machine_list,
    total_params=total_params,
    x_batch=single_samples
)
#print(grad, loss_mean, E_L_mean)

In [ ]:
import jax
import jax.numpy as jnp

# 正确展平 NNX 梯度，无报错
grad_values = jax.tree_util.tree_map(lambda x: x, grad)
grad_flat, _ = jax.tree_util.tree_flatten(grad_values)

# 计算总梯度范数（判断是否全0）
total_norm = jnp.linalg.norm(jnp.array([jnp.linalg.norm(g) for g in grad_flat]))

print("="*50)
print("🔍 梯度全零校验")
print("="*50)
print(f"总梯度范数: {total_norm:.8f}")
print(f"是否全零梯度: {total_norm < 1e-10}")

In [ ]:
import time
from NES_VMC import E_fcis,mcmc_sampler_multichain
# ======================
# 超参数
# ======================
N_CHAINS = 16
N_WARMUP = 50
N_SAMPLES_PER_CHAIN = 200
SWEEP_SIZE = 20
N_ITER =500

rngs = nnx.Rngs(42)
total_ansatz = NESTotalAnsatz(4, n_states=K, hidden_dim=8, rngs=rngs)
single_ansatz = SingleStateAnsatz(4, hidden_dim=8, rngs=rngs)
total_machine, total_graphdef, total_params = create_machine(total_ansatz)
total_matrix_machine, total_graphdef, total_params = create_machine_matrix(total_ansatz)

single_machine_list = []
for ansatz in total_ansatz.single_ansatz_list:
    m, g, p = create_single_machine(ansatz)
    single_machine_list.append(m)
    
optimizer = optax.sgd(learning_rate=0.01)
opt_state = optimizer.init(total_params)

# ===================== 7. 训练循环（多链版本） =====================
print("\n" + "="*60)
print("开始多链 NES-VMC 训练 (自然梯度下降法)")

print("超参数")
print("="*60)

history = {
    'step': [],
    'energy': [],
    'energy_std': [],
    'loss': [],
    'params': [],
    'E_Lmatrix':[],
    'natural_grad':[],
    'grad_flat':[],
    'samples':[],
    'log_Psi':[],
    'log_M':[]
}
print(f"基态能量={E_fcis[0]:.8f} Ha| 第一激发态能量={E_fcis[1]:.8f} Ha| 第二激发态能量={E_fcis[2]:.8f} Ha")
sampler_state = init_sampler_state(hi_ext, N_CHAINS, seed=21)  # 每次迭代换种子避免初始状态固定
start_time = time.time()
for step in range(N_ITER):
    # 1. 生成多链随机初始状态（模仿NetKet，无需手动指定单个initial_state）
    # 2. 多链采样（总样本数=16*63=1008，和原单链一致）
    samples,sampler_state = mcmc_sampler_multichain(
        n_samples_per_chain=N_SAMPLES_PER_CHAIN,
        n_warmup=N_WARMUP,
        sampler_state=sampler_state,
        edges=((0,1),(2,3),(4,5),(6,7)),
        machine=total_machine,
        params=total_params,
    )
    #samples = samples.reshape(-1,2,4)

    # 3. 计算能量和自然梯度（逻辑和原代码一致）
    grad, loss_mean, E_L_mean = nes_vmc_gradient(ha=ha,
                                                 total_matrix_machine=total_matrix_machine,
                                                 total_machine=total_machine,
                                                 single_machine_list=single_machine_list,
                                                 total_params=total_params,
                                                 x_batch=samples.reshape(-1,K,4))
    grad = jax.tree_util.tree_map(lambda x: x * 2, grad)
    #model_output = log(\Psi(X)) 
    # qgt_reg,qgt_unravel_fun = compute_nes_qgt(total_machine, total_params, samples.reshape(-1,K,4), diag_shift=0.01) 
    grad_flat , grad_unravel_fn = ravel_pytree(grad)
    
    # # # 自然梯度求解
    # natural_grad = jnp.linalg.solve(qgt_reg, grad_flat)
    # natural_grad = grad_unravel_fn(natural_grad)
    # grad = natural_grad
        
    # 4. 更新参数
    updates, opt_state = optimizer.update(grad, opt_state, total_params)
    total_params = optax.apply_updates(total_params, updates)
    
    # 5. 记录历史
    if step % 20 == 0 or step == N_ITER - 1:
        # total_model =  nnx.merge(graphdef,total_params)
        # log_Psi,log_M  = total_model(samples.reshape(-1,2,4))
        eig_vals, eig_vecs = jnp.linalg.eigh(E_L_mean)
        history['step'].append(step)
        history['E_Lmatrix'].append(E_L_mean)
        history['samples'].append(samples)
        history['loss'].append(loss_mean)
        # #history['natural_grad'].append(natural_grad)
        # history['grad_flat'].append(grad_flat)
        # history['log_Psi'].append(log_Psi)
        # history['log_M'].append(log_M)
        history['params'].append(total_params)
        print(f"Step {step:3d} | Loss: {loss_mean}|基态能量={eig_vals[0]:.8f} Ha| 第一激发态能量={eig_vals[1]:.8f} Ha Ha")
        print(f'grad={grad_flat[30:31]}')


end_time = time.time()
print(f"训练耗时：{end_time - start_time:.2f} 秒")
# 最终结果
print("\n" + "="*60)
print(f"训练完成!")
# print(f"最终能量：{final_energy.real:.8f} ± {final_std:.6f} Ha")
# print(f"FCI 基准：{E_fcis[0]:.8f} Ha")
# print(f"绝对误差：{final_error:.6f} Ha")
# print(f"相对误差：{final_error / jnp.abs(E_fcis[0]) * 100:.4f}%")
print("="*60)

In [ ]:
history['E_Lmatrix'][0]
eig_vals, eig_vecs = jnp.linalg.eigh(history['E_Lmatrix'][0])
eig_vals

In [ ]:
import time
from NES_VMC import E_fcis,mcmc_sampler_multichain
# ======================
# 超参数
# ======================
N_CHAINS = 16
N_WARMUP = 50
N_SAMPLES_PER_CHAIN = 200
SWEEP_SIZE = 20
N_ITER =500

rngs = nnx.Rngs(42)
total_ansatz = NESTotalAnsatz(4, n_states=K, hidden_dim=8, rngs=rngs)
single_ansatz = SingleStateAnsatz(4, hidden_dim=8, rngs=rngs)
total_machine, total_graphdef, total_params = create_machine(total_ansatz)
total_matrix_machine, total_graphdef, total_params = create_machine_matrix(total_ansatz)

single_machine_list = []
for ansatz in total_ansatz.single_ansatz_list:
    m, g, p = create_single_machine(ansatz)
    single_machine_list.append(m)
    
optimizer = optax.sgd(learning_rate=0.01)
opt_state = optimizer.init(total_params)

# ===================== 7. 训练循环（多链版本） =====================
print("\n" + "="*60)
print("开始多链 NES-VMC 训练 (自然梯度下降法)")

print("超参数")
print("="*60)

history = {
    'step': [],
    'energy': [],
    'energy_std': [],
    'loss': [],
    'params': [],
    'E_Lmatrix':[],
    'natural_grad':[],
    'grad_flat':[],
    'samples':[],
    'log_Psi':[],
    'log_M':[]
}
print(f"基态能量={E_fcis[0]:.8f} Ha| 第一激发态能量={E_fcis[1]:.8f} Ha| 第二激发态能量={E_fcis[2]:.8f} Ha")
sampler_state = init_sampler_state(hi_ext, N_CHAINS, seed=21)  # 每次迭代换种子避免初始状态固定
start_time = time.time()
for step in range(N_ITER):
    # 1. 生成多链随机初始状态（模仿NetKet，无需手动指定单个initial_state）
    # 2. 多链采样（总样本数=16*63=1008，和原单链一致）
    samples,sampler_state = mcmc_sampler_multichain(
        n_samples_per_chain=N_SAMPLES_PER_CHAIN,
        n_warmup=N_WARMUP,
        sampler_state=sampler_state,
        edges=((0,1),(2,3),(4,5),(6,7)),
        machine=total_machine,
        params=total_params,
    )
    #samples = samples.reshape(-1,2,4)

    # 3. 计算能量和自然梯度（逻辑和原代码一致）
    grad, loss_mean, E_L_mean = nes_vmc_gradient(ha=ha,
                                                 total_matrix_machine=total_matrix_machine,
                                                 total_machine=total_machine,
                                                 single_machine_list=single_machine_list,
                                                 total_params=total_params,
                                                 x_batch=samples.reshape(-1,K,4))
    grad = jax.tree_util.tree_map(lambda x: x * 2, grad)
    #model_output = log(\Psi(X)) 
    qgt_reg,qgt_unravel_fun = compute_nes_qgt(total_machine, total_params, samples.reshape(-1,K,4), diag_shift=0.01) 
    grad_flat , grad_unravel_fn = ravel_pytree(grad)
    
    # # 自然梯度求解
    natural_grad = jnp.linalg.solve(qgt_reg, grad_flat)
    natural_grad = grad_unravel_fn(natural_grad)
    grad = natural_grad
        
    # 4. 更新参数
    updates, opt_state = optimizer.update(grad, opt_state, total_params)
    total_params = optax.apply_updates(total_params, updates)
    
    # 5. 记录历史
    if step % 5 == 0 or step == N_ITER - 1:
        # total_model =  nnx.merge(graphdef,total_params)
        # log_Psi,log_M  = total_model(samples.reshape(-1,2,4))
        eig_vals, eig_vecs = jnp.linalg.eigh(E_L_mean)
        history['step'].append(step)
        history['E_Lmatrix'].append(E_L_mean)
        history['samples'].append(samples)
        history['loss'].append(loss_mean)
        # #history['natural_grad'].append(natural_grad)
        # history['grad_flat'].append(grad_flat)
        # history['log_Psi'].append(log_Psi)
        # history['log_M'].append(log_M)
        history['params'].append(total_params)
        print(f"Step {step:3d} | Loss: {loss_mean}|基态能量={eig_vals[0]:.8f} Ha| 第一激发态能量={eig_vals[1]:.8f} Ha Ha")
        print(f'grad={grad_flat[30:31]}')


end_time = time.time()
print(f"训练耗时：{end_time - start_time:.2f} 秒")
# 最终结果
print("\n" + "="*60)
print(f"训练完成!")
# print(f"最终能量：{final_energy.real:.8f} ± {final_std:.6f} Ha")
# print(f"FCI 基准：{E_fcis[0]:.8f} Ha")
# print(f"绝对误差：{final_error:.6f} Ha")
# print(f"相对误差：{final_error / jnp.abs(E_fcis[0]) * 100:.4f}%")
print("="*60)

In [ ]:
eig_vals, eig_vecs = jnp.linalg.eigh(E_L_mean)
eig_vals